# AgentRL vs TRL: Fair GRPO + Systems Moat

This Colab mirrors `codeDemo.ipynb`, but separates two claims:

1. **Fairness track:** AgentRL standard rollout vs TRL after the same SFT bootstrap.
2. **Systems track:** AgentRL-only runtime modes show continuous batching, speculative decoding, and paged-KV telemetry.

The default model is `Qwen/Qwen2.5-1.5B-Instruct` because it is the practical T4 path.

In [ ]:
REPO_URL = "https://github.com/kalyaannnn/agentRL.git"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DRAFT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 0
LIMIT = 8
SFT_EPOCHS = 1
GRPO_STEPS = 3
BATCH_SIZE = 1
GROUP_SIZE = 4
MAX_NEW_TOKENS = 64
MAX_EPISODE_STEPS = 2
OUTPUT_DIR = "/content/agentrl_colab_demo"


In [ ]:
!cd /content && rm -rf agentRL
!cd /content && git clone https://github.com/kalyaannnn/agentRL.git
%cd /content/agentRL
!pip install -U pip
!pip install -e ".[benchmark]"
!pip install -U trl peft accelerate bitsandbytes datasets pandas


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Fair comparison track

Both stacks use the same records, supervised targets, SFT bootstrap, seed, generation budget, and single-turn verifier. AgentRL uses standard rollout here so the comparison is about the post-training pipeline, not AgentRL-specific runtime systems.

In [ ]:
!python -m examples.compare_single_turn_baselines \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --limit 8 \
  --seed 0 \
  --sft-epochs 1 \
  --steps 3 \
  --batch-size 1 \
  --group-size 4 \
  --max-new-tokens 64 \
  --output-dir /content/agentrl_colab_demo/fair_compare


In [ ]:
import json
from pathlib import Path
import pandas as pd

comparison = json.loads(Path('/content/agentrl_colab_demo/fair_compare/comparison.json').read_text())
pd.DataFrame([
    {
        'framework': result.get('framework', name),
        'model': result.get('model_name'),
        'sft_epochs': result.get('sft_epochs'),
        'grpo_steps': result.get('steps'),
        'mean_reward': result.get('mean_reward'),
        'wall_time_s': result.get('wall_time_s'),
        'peak_vram_mb': result.get('peak_vram_mb'),
    }
    for name, result in comparison.items()
    if isinstance(result, dict) and 'framework' in result
])


## AgentRL systems moat

This section is AgentRL-only. It compares runtime modes on the same model scale so the telemetry shows where rollout time and memory go. Speculative decoding uses a smaller Qwen draft model. Paged KV is a memory/control-path demo and may not be the fastest mode on very small T4 runs.

In [ ]:
!python -m examples.benchmark_systems \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --draft-model Qwen/Qwen2.5-0.5B-Instruct \
  --task tool-use \
  --split easy \
  --steps 3 \
  --batch-size 1 \
  --group-size 4 \
  --max-new-tokens 64 \
  --max-episode-steps 2 \
  --output-dir /content/agentrl_colab_demo/systems_compare \
  --compare-runtime-modes \
  --include-speculative


In [ ]:
systems = json.loads(Path('/content/agentrl_colab_demo/systems_compare/comparison.json').read_text())
runs = systems.get('runs') or systems.get('summaries') or []
pd.DataFrame([
    {
        'mode': run.get('mode_name'),
        'mean_step_time_ms': run.get('mean_step_time_ms'),
        'tokens_per_second': run.get('mean_tokens_per_second'),
        'peak_vram_mb': run.get('peak_vram_mb'),
        'rollout_peak_vram_mb': run.get('rollout_peak_vram_mb'),
        'padding_ratio': run.get('mean_padding_ratio'),
        'kv_pressure': run.get('mean_scheduler_kv_pressure'),
        'bottleneck': run.get('dominant_runtime_bottleneck'),
        'reward': run.get('mean_reward'),
    }
    for run in runs
])


## Interpretation

- TRL is the fair algorithmic baseline after the same SFT bootstrap.
- AgentRL's moat is the runtime layer: continuous batching, speculative decoding, paged-KV telemetry, scheduler diagnostics, VRAM/headroom metrics, and rollout throughput.
- For A100/L4, switch `MODEL` to `Qwen/Qwen2.5-7B-Instruct` and use `Qwen/Qwen2.5-1.5B-Instruct` as the draft model for speculative experiments.